# Notebook 4 – FEINA Sample Preparation

Select a random sample of 346 segments from the FEINA dataset to use
as an external evaluation set. 


**Output:** `feina_sample_346.csv` — ready to use as input for Notebooks 3a and 3b.

---

### Expected data directory structure

```
data/
└── feina/
    └── Dataset_FEINA_10122023.xlsx   ← original FEINA dataset
```

## Installation

In [ ]:
!pip install pandas openpyxl nltk

## 1. Load and clean the FEINA dataset

In [ ]:
import pandas as pd
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

FEINA_PATH  = 'data/feina/Dataset_FEINA_10122023.xlsx'
OUTPUT_PATH = 'data/feina/feina_sample_346.csv'
SAMPLE_SIZE = 346
RANDOM_SEED = 42     # fixed seed for reproducibility

df = pd.read_excel(FEINA_PATH)
df = df.rename(columns={'Segmento': 'original_text', 'Propuesta': 'simplified_text'})

df = df.dropna(subset=['original_text', 'simplified_text']).copy()
df['original_text']   = df['original_text'].astype(str).str.strip()
df['simplified_text'] = df['simplified_text'].astype(str).str.strip()
df = df[
    (df['original_text'].str.len() > 0) &
    (df['simplified_text'].str.len() > 0)
].reset_index(drop=True)

print(f'Total clean segments in FEINA: {len(df)}')
df.head(3)

## 2. Random sample selection

In [ ]:
sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

sample['document_id']          = 'FEINA'
sample['original_sentence_id'] = sample.index

sample = sample[['document_id', 'original_sentence_id', 'original_text', 'simplified_text']]

print(f'Sample size: {len(sample)} segments')
print(f'Random seed: {RANDOM_SEED}')
sample.head(5)

## 3. Basic statistics of the sample

In [ ]:
def count_tokens(text):
    return len(word_tokenize(str(text), language='spanish'))

sample['orig_tokens'] = sample['original_text'].apply(count_tokens)
sample['simp_tokens'] = sample['simplified_text'].apply(count_tokens)
sample['ratio']       = sample['simp_tokens'] / sample['orig_tokens']

stats = sample[['orig_tokens', 'simp_tokens', 'ratio']].describe(
    percentiles=[0.25, 0.5, 0.75]).round(2)
stats.index = ['count','mean','std','min','Q1','median','Q3','max']
print('FEINA sample statistics:')
print(stats.to_string())
print(f"\nMean simplification ratio (simp/orig): {sample['ratio'].mean():.3f}")
print(f"Ratio > 1 in {(sample['ratio'] > 1).sum()}/{len(sample)} segments")

## 4. Save the sample

In [ ]:
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

sample[['document_id', 'original_sentence_id', 'original_text', 'simplified_text']].to_csv(
    OUTPUT_PATH, index=False)

print(f'Sample saved to: {OUTPUT_PATH}')
print(f'Segments: {len(sample)}')